# 🏢 Enterprise Orchestration: GRPO Training with Unsloth

This notebook demonstrates the real reinforcement learning pipeline for the **Enterprise Orchestration Environment**. It uses `TRL`'s `GRPOTrainer` and `Unsloth` to optimize the `Qwen2.5-1.5B-Instruct` model directly against the environment's reward signals.

**Note:** To run this notebook, you need a GPU (T4 is sufficient thanks to Unsloth 4-bit quantization).

### 1. Install Dependencies

In [ ]:
!pip install -q openenv-core
!pip install -q "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets matplotlib

### 2. Setup Environment

In [ ]:
import os
if not os.path.exists('scaler-final'):
    !git clone https://github.com/redhatsam09/scaler-final.git
import sys
sys.path.append('/content/scaler-final')

import json
import torch
from datasets import Dataset
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig

from src.environment import DataCleaningEnv
from src.graders import EnterpriseOrchestrationGrader
from src.models import Action

### 3. Define the World Model Environment Reward
This is where the magic happens. We define a custom reward function that runs the LLM's chosen action against the actual environment.

In [ ]:
def build_prompt(obs):
    return f"""You are an enterprise workflow orchestrator managing CRM, Billing, and Support systems.
Given the current state, output a JSON action with:
- action_type: one of [analyze, impute, deduplicate, validate, report_findings, delegate, resolve_alert, reconcile_apps, oversight_review, inspect_actor, audit_records, request_policy_clarification]
- target_columns: list of column names
- parameters: dict of action parameters
- reasoning: why you chose this action

Respond ONLY with valid JSON.
Current State:
{obs.natural_language_observation}
Dataset: {obs.dataset_shape[0]} rows, {obs.dataset_shape[1]} cols
KPIs: {json.dumps(obs.kpi_snapshot)}
Urgency: {obs.urgency_signals if obs.urgency_signals else 'None'}
What action should you take?"""

def environment_reward_function(completions, prompts, **kwargs):
    import re
    rewards = []
    env = DataCleaningEnv(seed=42)
    
    for i, completion in enumerate(completions):
        try:
            obs = env.reset(task_id="task_enterprise_orchestration", seed=1000 + i)
            action_data = None
            match = re.search(r'\{.*\}', completion, re.DOTALL)
            if match:
                action_data = json.loads(match.group())
            
            if action_data:
                action = Action(
                    action_type=action_data.get("action_type", "analyze"),
                    target_columns=action_data.get("target_columns", obs.column_names[:3]),
                    parameters=action_data.get("parameters", {}),
                    reasoning=action_data.get("reasoning", "")
                )
                _, reward, _, _ = env.step(action)
                grade = EnterpriseOrchestrationGrader.grade(env.current_episode)
                combined = 0.6 * reward.value + 0.4 * float(grade)
                rewards.append(min(1.0, max(-1.0, combined + 0.1))) # Format bonus
            else:
                rewards.append(-0.5) # Invalid JSON
        except Exception:
            rewards.append(-0.5)
    return rewards

### 4. Load Model and Generate Rollout Prompts

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

# Create training dataset of 50 environment states
env = DataCleaningEnv(seed=42)
prompts = [build_prompt(env.reset(task_id="task_enterprise_orchestration", seed=2000+i)) for i in range(50)]
dataset = Dataset.from_dict({"prompt": prompts})

### 5. Run GRPO Optimization

In [ ]:
config = GRPOConfig(
    output_dir="/content/grpo_checkpoint",
    num_generations=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_steps=30,
    logging_steps=5,
    save_strategy="no",
    max_completion_length=512,
    report_to=[],
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    train_dataset=dataset,
    reward_funcs=[environment_reward_function],
)

result = trainer.train()
print(f"Training complete. Final Loss: {result.training_loss}")